# Moving-Average Cross Meta-Labeling

Build a trend-following primary model from moving-average crosses, derive AFML meta-labels with `pt_sl=[1, 2]` and a one-day vertical barrier, then train a random forest to decide whether to trade. The primary model decides the side; the random forest only decides bet/no bet.

In [ ]:
from pathlib import Path
import subprocess
import sys

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, balanced_accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import TimeSeriesSplit

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from afml.bars import dollar_bars
from afml.data import read_tick_csv
from afml.labeling import add_vertical_barrier, drop_labels, get_bins, get_events
from afml.sampling import daily_volatility, target_from_events
from afml.strategies import moving_average_cross_side

In [ ]:
RAW_PATH = PROJECT_ROOT / "data" / "Binance" / "spot" / "monthly" / "trades" / "BTCUSDT" / "BTCUSDT-trades-2026-05.csv"
NORMALIZED_PATH = PROJECT_ROOT / "data" / "processed" / "binance" / "trades" / "BTCUSDT" / "BTCUSDT-trades-2026-05-head4m-normalized.csv"
MAX_RAW_ROWS = 4_000_000

if not NORMALIZED_PATH.exists():
    subprocess.run(
        [
            sys.executable,
            str(PROJECT_ROOT / "scripts" / "prepare_binance_trades.py"),
            str(RAW_PATH),
            "-o",
            str(NORMALIZED_PATH),
            "--max-rows",
            str(MAX_RAW_ROWS),
        ],
        cwd=PROJECT_ROOT,
        check=True,
    )

NORMALIZED_PATH

In [ ]:
NROWS = None
DOLLAR_BAR_COUNT = 500

raw_ticks = read_tick_csv(NORMALIZED_PATH, nrows=NROWS)
dollar_threshold = raw_ticks["dollar_value"].sum() / DOLLAR_BAR_COUNT
bars = dollar_bars(raw_ticks, threshold=dollar_threshold)
duplicate_bar_timestamps = bars.index.duplicated().sum()
bars = bars[~bars.index.duplicated(keep="last")]
close = bars["close"]

pd.Series(
    {
        "ticks": len(raw_ticks),
        "bars": len(bars),
        "duplicate_bar_timestamps_dropped": duplicate_bar_timestamps,
        "start": close.index.min(),
        "end": close.index.max(),
        "dollar_threshold": dollar_threshold,
    }
)

In [ ]:
FAST_WINDOW = 20
SLOW_WINDOW = 50

side = moving_average_cross_side(close, fast_window=FAST_WINDOW, slow_window=SLOW_WINDOW)
t_events = side.index

side.value_counts().rename("count")

In [ ]:
daily_vol = daily_volatility(close, span=100)

if daily_vol.empty:
    raise ValueError(
        "daily_volatility is empty. Snippet 3.1 needs observations at least one calendar day apart; "
        "load multiple days of bars before deriving these meta-labels."
    )

trgt = target_from_events(daily_vol, t_events, min_ret=0)
vertical_barriers = add_vertical_barrier(trgt.index, close, num_days=1)

meta_events = get_events(
    close=close,
    t_events=trgt.index,
    pt_sl=[1, 2],
    trgt=trgt,
    min_ret=0,
    t1=vertical_barriers,
    side=side,
)
labels = get_bins(meta_events, close)
labels = drop_labels(labels, min_pct=0.05)

labels["bin"].value_counts(normalize=True).rename("class_fraction")

In [ ]:
returns = close.pct_change()
fast_ma = close.rolling(FAST_WINDOW).mean()
slow_ma = close.rolling(SLOW_WINDOW).mean()
ma_spread = fast_ma / slow_ma - 1


def rolling_zscore(series: pd.Series, window: int) -> pd.Series:
    mean = series.rolling(window).mean()
    std = series.rolling(window).std()
    return (series - mean) / std.replace(0, np.nan)


def run_length(series: pd.Series) -> pd.Series:
    groups = series.ne(series.shift()).cumsum()
    return series.groupby(groups).cumcount() + 1


features = pd.DataFrame(index=close.index)

# Primary-model state: is the MA-cross side strong, fresh, or unstable?
features["side"] = side
features["ma_spread"] = ma_spread
features["ma_spread_abs"] = ma_spread.abs()
features["ma_spread_z20"] = rolling_zscore(ma_spread, 20)
features["ma_spread_chg"] = ma_spread.diff()
features["side_age"] = run_length(side)
features["side_flip_20"] = side.ne(side.shift()).astype(float).rolling(20).sum()

# Trend quality: has recent realized path agreed with the proposed side?
signed_direction = np.sign(returns).where(returns.notna())
features["ret_1"] = returns
features["ret_5"] = returns.rolling(5).sum()
features["ret_20"] = returns.rolling(20).sum()
features["side_ret_1"] = returns * side
features["side_ret_5"] = returns.rolling(5).sum() * side
features["side_ret_20"] = returns.rolling(20).sum() * side
features["trend_hit_rate_20"] = signed_direction.eq(side).astype(float).where(returns.notna()).rolling(20).mean()

# Volatility regime: meta-labels often depend on whether barriers are realistically reachable.
features["daily_vol"] = daily_vol.reindex(close.index, method="ffill")
features["daily_vol_z20"] = rolling_zscore(features["daily_vol"], 20)
features["realized_vol_10"] = returns.rolling(10).std()
features["realized_vol_20"] = returns.rolling(20).std()
features["vol_ratio_10_20"] = features["realized_vol_10"] / features["realized_vol_20"]

# Bar shape: compression, impulse, and wick structure from OHLC bars.
bar_range = bars["high"] / bars["low"] - 1
body = (bars["close"] - bars["open"]).abs()
features["bar_return"] = bars["close"] / bars["open"] - 1
features["bar_range"] = bar_range
features["body_to_range"] = body / (bars["high"] - bars["low"]).replace(0, np.nan)
features["upper_wick"] = (bars["high"] - bars[["open", "close"]].max(axis=1)) / bars["close"]
features["lower_wick"] = (bars[["open", "close"]].min(axis=1) - bars["low"]) / bars["close"]

# Liquidity / bar speed: a dollar bar that forms quickly is a different regime from a slow one.
bar_duration_sec = bars.index.to_series().diff().dt.total_seconds()
features["bar_duration_sec"] = bar_duration_sec
features["log_tick_count"] = np.log1p(bars["tick_count"])
features["log_volume"] = np.log1p(bars["volume"])
features["log_dollar_value"] = np.log1p(bars["dollar_value"])
features["duration_z20"] = rolling_zscore(bar_duration_sec, 20)
features["tick_count_z20"] = rolling_zscore(features["log_tick_count"], 20)

# Shift once at the end so every feature is known before the event starts.
features = features.replace([np.inf, -np.inf], np.nan).shift(1)

sample = features.join(labels["bin"], how="inner").dropna()
X = sample.drop(columns="bin")
y = sample["bin"].astype(int)

pd.Series({"samples": len(sample), "features": X.shape[1], "positive_fraction": y.mean()})


In [ ]:
if y.nunique() < 2:
    raise ValueError("Need at least two meta-label classes after filtering to train the classifier.")

rf_params = dict(
    n_estimators=500,
    max_depth=4,
    min_samples_leaf=3,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1,
)

# Keep an end-of-sample split for a familiar sanity check.
split = int(len(X) * 0.7)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

if y_train.nunique() < 2:
    raise ValueError("The time split left the training set with only one class; use more data or a different split.")

clf = RandomForestClassifier(**rf_params)
clf.fit(X_train, y_train)

positive_class_col = list(clf.classes_).index(1)
pred = pd.Series(clf.predict(X_test), index=X_test.index)
proba = pd.Series(clf.predict_proba(X_test)[:, positive_class_col], index=X_test.index)

metrics = {
    "train_samples": len(X_train),
    "test_samples": len(X_test),
    "test_positive_fraction": y_test.mean(),
    "baseline_always_trade_accuracy": y_test.mean(),
    "accuracy": (pred == y_test).mean(),
}
if y_test.nunique() == 2:
    metrics["balanced_accuracy"] = balanced_accuracy_score(y_test, pred)
    metrics["roc_auc"] = roc_auc_score(y_test, proba)
    metrics["average_precision"] = average_precision_score(y_test, proba)
else:
    print("End-of-sample window has one label only; accuracy is not a useful skill measure here.")

print(pd.Series(metrics))
print(confusion_matrix(y_test, pred, labels=[0, 1]))
print(classification_report(y_test, pred, labels=[0, 1], zero_division=0))

# A tiny walk-forward diagnostic is more informative than trusting one terminal window.
fold_rows = []
for fold, (train_idx, test_idx) in enumerate(TimeSeriesSplit(n_splits=5).split(X), start=1):
    X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
    y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
    if y_tr.nunique() < 2:
        continue

    fold_clf = RandomForestClassifier(**rf_params)
    fold_clf.fit(X_tr, y_tr)
    fold_pred = pd.Series(fold_clf.predict(X_te), index=X_te.index)
    fold_proba = pd.Series(fold_clf.predict_proba(X_te)[:, list(fold_clf.classes_).index(1)], index=X_te.index)

    row = {
        "fold": fold,
        "train_samples": len(X_tr),
        "test_samples": len(X_te),
        "test_positive_fraction": y_te.mean(),
        "accuracy": (fold_pred == y_te).mean(),
        "balanced_accuracy": np.nan,
        "roc_auc": np.nan,
        "average_precision": np.nan,
    }
    if y_te.nunique() == 2:
        row["balanced_accuracy"] = balanced_accuracy_score(y_te, fold_pred)
        row["roc_auc"] = roc_auc_score(y_te, fold_proba)
        row["average_precision"] = average_precision_score(y_te, fold_proba)
    fold_rows.append(row)

walk_forward = pd.DataFrame(fold_rows)
walk_forward


In [ ]:
feature_importance = pd.Series(clf.feature_importances_, index=X.columns).sort_values(ascending=False)
feature_importance.head(20)
